# E06：NumPy 数组与向量化计算

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：验证性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 从结构化记录中提取数值字段并创建 NumPy 数组。
2. 检查数组的维数、形状、元素数量和数据类型。
3. 使用索引、切片和向量化运算完成批量数值计算。
4. 比较循环与向量化写法，并使用形状和数值检查核验结果。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查数组形状、样本量和统计口径。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- 数组计算结果异常时，先检查 `shape`、缺失值和布尔掩码。
- 如果变量状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 输入检查与数组创建：15 分钟
- 形状、索引和切片：15 分钟
- 标量运算与数组运算：15 分钟
- 循环和向量化对照：15 分钟
- 二维数组与结果核验：15 分钟
- 调试、实验小结与提交：15 分钟

## 主线输入：个人成果优先、官方检查点兜底

E06–E07 延续共享数据主线：优先读取学生自己的 E05 输出，文件不存在时使用官方检查点。输入准备代码会统一核验记录数、字段、主键和缺失数量；本实验重点是从记录结构转向 NumPy 数组。E06 先使用非缺失记录，E07 再正式处理 `np.nan`。

输入按以下顺序查找：

1. E05 标准输出目录中的个人成果。
2. 当前工作目录中的 `e05_cleaned_records.json`，按学生自行提供的输入处理，来源需自行确认。
3. 仓库中的官方 E05 检查点。
4. 当前工作目录中的 `e05_cleaned_records_official.json` 官方副本。
5. 联网获取官方检查点，核验通过后按上述官方副本名称缓存到当前工作目录。

将 `USE_STUDENT_E05_OUTPUT` 改为 `False` 时，跳过前两项。只有文件不存在才会继续向后查找；已有文件解析或核验失败会明确报错，不会静默改用官方数据。以准备单元打印的来源、实际路径和当前工作目录为准；若从其他目录启动内核，当前工作目录不一定是 Notebook 所在目录。

准备代码直接运行即可，不要求掌握联网下载实现。单独下载 Notebook 时，可自行提供同名输入或联网取得官方副本；官方副本另取文件名，避免覆盖或误认个人成果。

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

import numpy as np

# True：优先使用个人成果；False：跳过个人文件，直接使用官方检查点。
USE_STUDENT_E05_OUTPUT = True

course_lab_dir = Path("..")
if not (course_lab_dir / "chapter2").exists():
    course_lab_dir = Path("lab/applied-stat-64")

shared_dir = Path("../../shared")
if not shared_dir.exists():
    shared_dir = Path("lab/shared")

student_path = course_lab_dir / "chapter2" / "e05_output" / "e05_cleaned_records.json"
current_student_path = Path("e05_cleaned_records.json")
official_path = shared_dir / "checkpoints" / "e05_cleaned_records.json"
# 官方本地副本另取文件名，避免与学生提供的同名输入混淆。
official_cache_path = Path("e05_cleaned_records_official.json")

input_candidates = []
if USE_STUDENT_E05_OUTPUT:
    input_candidates.extend([
        (student_path, "学生 E05 标准输出"),
        (current_student_path, "当前工作目录自行提供的 E05 输入（来源需自行确认）"),
    ])
input_candidates.extend([
    (official_path, "仓库官方 E05 检查点"),
    (official_cache_path, "当前工作目录官方检查点副本"),
])

input_path = None
downloaded_text = None
for candidate_path, candidate_source in input_candidates:
    if candidate_path.exists():
        input_path = candidate_path
        input_source = candidate_source
        break

if input_path is None:
    input_path = official_cache_path
    input_source = "联网获取的官方 E05 检查点"
    official_url = (
        "https://rwang0040.github.io/python-for-statistics-public/"
        "lab/shared/checkpoints/e05_cleaned_records.json"
    )
    print("本地未找到可用路径，尝试联网获取官方检查点：", official_url)
    try:
        with urlopen(official_url, timeout=10) as response:
            downloaded_text = response.read().decode("utf-8")
    except OSError as error:
        raise RuntimeError(
            "无法联网获取官方检查点。请检查网络，或手动下载官方文件，"
            "重命名为 e05_cleaned_records_official.json 后放到当前工作目录。"
        ) from error

print("输入来源：", input_source)
print("输入文件：", input_path.resolve())
print("当前工作目录：", Path.cwd())

# 只有文件不存在才向后查找；已有文件损坏或核验失败时不静默切换来源。
if downloaded_text is None:
    with open(input_path, "r", encoding="utf-8") as file:
        records = json.load(file)
else:
    records = json.loads(downloaded_text)

if not isinstance(records, list) or not all(isinstance(record, dict) for record in records):
    raise ValueError("输入必须是由字典记录组成的列表，请检查以上输入来源和路径。")

expected_fields = [
    "student_id",
    "alias",
    "gender",
    "major",
    "prior_programming",
    "support_plan",
    "reported_study_hours",
    "reported_attendance_rate",
]
student_ids = [record.get("student_id") for record in records]
missing_hours = sum(
    record.get("reported_study_hours") is None
    for record in records
)
missing_attendance = sum(
    record.get("reported_attendance_rate") is None
    for record in records
)

input_checks = {
    "记录数为 160": len(records) == 160,
    "student_id 唯一且非空": (
        all(isinstance(value, str) and bool(value) for value in student_ids)
        and len(set(student_ids)) == len(student_ids)
    ),
    "字段及顺序一致": all(
        list(record.keys()) == expected_fields
        for record in records
    ),
    "学习时长缺失数为 8": missing_hours == 8,
    "出勤率缺失数为 4": missing_attendance == 4,
}

for check_name, passed in input_checks.items():
    print(check_name, "：", passed)

if not all(input_checks.values()):
    raise ValueError(
        "输入核验未通过：请检查以上来源和文件。个人成果有误时请先修正；"
        "如需主动改用官方数据，可把 USE_STUDENT_E05_OUTPUT 改为 False。"
    )

# 下载内容通过同一套核验后才保存；网络错误与本地写入错误分别提示。
if downloaded_text is not None:
    try:
        official_cache_path.write_text(downloaded_text, encoding="utf-8")
    except OSError as error:
        raise RuntimeError(
            "官方数据已获取并通过核验，但无法保存本地副本。"
            "请检查当前工作目录的写入权限或磁盘空间。"
        ) from error
    print("官方检查点已缓存：", official_cache_path.resolve())

## 任务 1：从记录中创建一维数组

提取所有非缺失的 `reported_study_hours`，创建浮点数组 `study_hours`。输出数组类型、`ndim`、`shape`、`size` 和 `dtype`，并核对有效记录数。

In [ ]:
# TODO：提取非缺失学习时长并创建浮点数组
study_hours = np.array([], dtype=float)

# TODO：输出 type、ndim、shape、size 和 dtype

## 任务 2：使用索引和切片观察数组

取得第一个、最后一个、前 5 个和每隔 20 个位置的学习时长。说明切片结果仍然是数组，并检查其形状。

In [ ]:
# TODO：取消注释并完成索引和切片
# print("第一个值：", study_hours[0])
# print("最后一个值：", study_hours[-1])
# print("前 5 个值：", study_hours[:5])
# print("每隔 20 个位置：", study_hours[::20])
# print("前 5 个值的 shape：", study_hours[:5].shape)

## 任务 3：完成向量化批量计算

把每周学习时长换算为 16 周总学习时长；再建立同形状的 8 小时目标数组，计算每条记录与目标的差值。检查输入和输出形状是否一致。

In [ ]:
# TODO：使用向量化运算计算 16 周总时长
semester_hours = study_hours * 0

# TODO：建立同形状的 8 小时目标数组并计算差值
weekly_targets = np.zeros(study_hours.shape)
hours_gap = study_hours - weekly_targets

# TODO：输出前 5 个结果并比较 shape

## 任务 4：比较循环和向量化写法

使用循环再次计算 16 周总时长，再与向量化结果并排比较。这里不做性能竞赛；检查两个结果的类型、形状和前 5 个值，并说明哪一种更直接表达“整组数值同时计算”。

In [ ]:
semester_hours_loop = []

# TODO：使用循环计算 16 周总时长，再转换为数组

# TODO：并排输出两个结果的类型、shape 和前 5 个值

## 任务 5：创建二维数值数组

筛选学习时长和出勤率都不缺失的记录，按“每行一名学生、两列分别为学习时长和出勤率”创建 `numeric_matrix`。检查整体形状、第一行和两列数据。

In [ ]:
# TODO：筛选两个数值字段都不缺失的记录
complete_records = []

# TODO：创建“记录数 × 2”的浮点数组
numeric_matrix = np.empty((0, 2), dtype=float)

print("二维数组 shape：", numeric_matrix.shape)
# TODO：观察第一行以及两列的前 5 个值

## 任务 6：核验数组口径

分别核验原始记录数、有效学习时长数、完整数值记录数和二维数组列数。再并排检查学习时长列与从完整记录重新提取结果的形状和前 5 个值。

In [ ]:
# TODO：从 complete_records 再次提取学习时长用于核验
hours_from_complete_records = np.array([], dtype=float)

print("原始记录数：", len(records))
print("有效学习时长数：", study_hours.size)
print("完整数值记录数：", numeric_matrix.shape[0])
print("二维数组列数：", numeric_matrix.shape[1])
# TODO：并排输出两个结果的 shape 和前 5 个值

## 实验小结

请用 Markdown 回答：

1. Python 列表和 NumPy 数组在“整体乘以 16”时有什么差别？
2. 为什么从多条记录构造二维数组时必须保持字段顺序和记录对齐？
3. 本实验使用非缺失记录，可能使样本量发生什么变化？

### 实验小结

- 